In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch, PathPatch
import seaborn as sns
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

font_path = "font-folder/Roboto-Light.ttf"
font_path_bold = "font-folder/Roboto-Bold.ttf"
font_prop = fm.FontProperties(fname=font_path, size=18)
font_prop_legend = fm.FontProperties(fname=font_path, size=14)
font_prop_numbers = fm.FontProperties(fname=font_path, size=10)
font_prop_title = fm.FontProperties(fname=font_path_bold, size=22)

In [ ]:
# ============================================================
# MODEL SETTINGS
# ============================================================
tsfm_models = ['Chronos-2', 'Lag-Llama', 'Moirai-2',
                'TabPFN-TS', 'TimeGPT', 'TimesFM']

model_type_color = {
    'Chronos-2':  "#4ECED4",
    'Moirai-2':   "#967ECD",
    'Lag-Llama': "#3E977E",
    'TabPFN-TS': "#CE5858",
    'TimeGPT':   "#1F77B4",
    'TimesFM':   "#F2B134",
    'hub-ensemble':  "teal",
    'hub-baseline': "#807B6DFF"
}

metric        = "ae_median_scaled_relative_skill"
metric_label  = "AE"

# ============================================================
# LOAD LOCATIONS MAP
# ============================================================
locations_df = pd.read_csv('https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/main/auxiliary-data/locations.csv')
loc_map = dict(zip(locations_df['location'].astype(str).str.zfill(2),
                   locations_df['abbreviation']))

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def load_boxplot_data(challenge):
    df = pd.read_csv(f"../model-evaluation/forecast_pairwise_comparison_scores/forecast_scores_pairwise_comparison_summary_log_{challenge}.csv")  
    df = df[["model", "location", "wis_scaled_relative_skill", "ae_median_scaled_relative_skill"]].drop_duplicates(subset=["model", "location"]).reset_index(drop=True)
    hub_models = tsfm_models + [ f'{challenge}-ensemble']
    df_tsfm = df[df['model'].isin(hub_models)].copy()
    df_tsfm['model_name'] = df_tsfm['model']
    df_tsfm['model_name'].replace({
        f'{challenge}-ensemble': 'hub-ensemble'
    }, inplace=True)
    models_median = df_tsfm.groupby("model_name")[metric].median().sort_values().index.tolist()
    return df_tsfm, models_median

def load_heatmap_data(challenge):
    df = pd.read_csv(f"../model-evaluation/forecast_pairwise_comparison_scores/forecast_scores_pairwise_comparison_summary_log_{challenge}.csv")
    df = df[["model", "location", "wis_scaled_relative_skill", "ae_median_scaled_relative_skill"]].drop_duplicates(subset=["model", "location"]).reset_index(drop=True)
    hub_models = tsfm_models + [f'{challenge}-baseline', f'{challenge}-ensemble']
    df_map = df[df['model'].isin(hub_models)].copy()
    df_map['model_name'] = df_map['model']
    df_map['model_name'].replace({
        f'{challenge}-baseline': 'hub-baseline',
        f'{challenge}-ensemble': 'hub-ensemble'
    }, inplace=True)
    df_pivot = df_map.pivot_table(index='location', columns='model_name', values=metric)
    model_order = df_pivot.median(skipna=True).sort_values().index
    df_pivot = df_pivot[model_order]
    df_pivot.index = [loc_map.get(str(l).zfill(2), str(l)) for l in df_pivot.index]
    df_pivot = df_pivot.sort_index()
    us_label = loc_map.get('US', 'US')
    order = [l for l in df_pivot.index if l != us_label] + [us_label]
    df_pivot = df_pivot.loc[order]
    return df_pivot  # trasponi: modelli su Y, locations su X

def plot_boxplot(ax, df_tsfm, models_median):
    model_colors = [model_type_color[m] for m in models_median]
    sns.boxplot(
        data=df_tsfm, x="model_name", y=metric,
        order=models_median, palette=model_colors,
        showfliers=False, medianprops=dict(color='white', linewidth=1.5),
        ax=ax
    )
    patches = [p for p in ax.patches if isinstance(p, PathPatch)]
    for i, patch in enumerate(patches):
        color = model_colors[i % len(model_colors)]
        patch.set_edgecolor(color)
        patch.set_linewidth(1.5)
    lines_per_box = len(ax.lines) // len(models_median)
    for i, line in enumerate(ax.lines):
        box_idx = i // lines_per_box
        if box_idx < len(model_colors):
            pos_in_box = i % lines_per_box
            if pos_in_box == lines_per_box - 1:
                line.set_color("white")
            else:
                line.set_color(model_colors[box_idx])
            line.set_linewidth(1.5)
    ax.axhline(y=1, linestyle="--", color="grey", linewidth=1)
    ax.set_xlabel("")
    ax.set_ylabel(f"Relative {metric_label}", fontproperties=font_prop)
    for label in ax.get_yticklabels() + ax.get_xticklabels():
        label.set_fontproperties(font_prop)
    for label in ax.get_xticklabels():
        label.set_rotation(45)

def plot_heatmap(ax, df_plot,add_cbar=True):
    norm = TwoSlopeNorm(vmin=0.5, vcenter=1.0, vmax=2.0)
    cmap = 'RdBu_r'
    sns.heatmap(
        df_plot, ax=ax, cmap=cmap, norm=norm,
        linewidths=1.2, linecolor='white',
        cbar=False, square=False,
        xticklabels=True, yticklabels=True,
        annot=True, fmt='.2f',
        annot_kws={'fontproperties': font_prop}
    )
    # Riquadra cella con WIS più basso per ogni stato (riga)
    for i in range(df_plot.shape[0]):
        row = df_plot.iloc[i]
        best_j = int(np.argmin(row.values))
        rect = plt.Rectangle(
            (best_j + 0.05, i + 0.05), 0.9, 0.9,  # leggermente dentro la cella
            fill=False, edgecolor='#333333', linewidth=1.7, zorder=5
        )
        ax.add_patch(rect)
    for label in ax.get_yticklabels():
        label.set_fontproperties(font_prop)
        label.set_rotation(0)
    for label in ax.get_xticklabels():
        label.set_fontproperties(font_prop)
        label.set_rotation(45)
        label.set_ha('right')
    ax.set_xlabel("")
    ax.set_ylabel("")
    if add_cbar:
        cbar = plt.colorbar(
            plt.cm.ScalarMappable(norm=norm, cmap=cmap),
            ax=ax, shrink=0.5, aspect=50, pad=0.02
        )
        cbar.set_label(f'Relative {metric_label}', fontproperties=font_prop)
        cbar.set_ticks([0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0])
        cbar.ax.axhline(y=1, color='black', linewidth=1.5, linestyle='--')
        for label in cbar.ax.get_yticklabels():
            label.set_fontproperties(font_prop)
    ax.set_xlabel("")
    ax.tick_params(axis='x', bottom=False)  # rimuovi tick marks

def plot_win_bar(ax_bar, df_plot, model_type_color):
    best_per_state = df_plot.idxmin(axis=1)  # per ogni stato, modello migliore
    win_counts = best_per_state.value_counts().reindex(df_plot.columns, fill_value=0)
    
    colors = [model_type_color.get(m, '#aaaaaa') for m in win_counts.index]
    ax_bar.bar(range(len(win_counts)), win_counts.values, color=colors, width=0.8)
    
    # Nomi modelli sull'asse X del barplot
    ax_bar.set_xticks(range(len(win_counts)))
    ax_bar.set_xticklabels(win_counts.index, rotation=30, ha='right')
    for label in ax_bar.get_xticklabels():
        label.set_fontproperties(font_prop_numbers)
    
    ax_bar.set_xlim(-0.5, len(win_counts) - 0.5)
    ax_bar.set_yticks([0, win_counts.max()])
    ax_bar.set_ylabel('N', fontproperties=font_prop_numbers, rotation=0, labelpad=10)
    for spine in ['top', 'right']:
        ax_bar.spines[spine].set_visible(False)
    for label in ax_bar.get_yticklabels():
        label.set_fontproperties(font_prop_numbers)
    for k, v in enumerate(win_counts.values):
        ax_bar.text(k, v + 0.1, str(int(v)), ha='center', va='bottom',
                    fontproperties=font_prop_numbers)
    ax_bar.set_ylim(0, win_counts.max() + 2)

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================
df_tsfm_flu, models_median_flu   = load_boxplot_data('FluSight')
df_tsfm_covid, models_median_covid = load_boxplot_data('CovidHub')
df_plot_flu   = load_heatmap_data('FluSight')
df_plot_covid = load_heatmap_data('CovidHub')

In [ ]:
df_plot_flu

In [ ]:
# ============================================================
# PANEL FIGURE
# ============================================================
from matplotlib.gridspec import GridSpecFromSubplotSpec
fig = plt.figure(figsize=(20, 32), dpi=200)
gs = fig.add_gridspec(1, 2)
ax_A = fig.add_subplot(gs[0, 0])
ax_B = fig.add_subplot(gs[0, 1])

plot_heatmap(ax_A, df_plot_flu,   add_cbar=True)
plot_heatmap(ax_B, df_plot_covid, add_cbar=True)

# Stesso ylim per entrambi i boxplot
ymin = min(ax_A.get_ylim()[0], ax_B.get_ylim()[0])
ymax = max(ax_A.get_ylim()[1], ax_B.get_ylim()[1])
ax_A.set_ylim(ymin, ymax)
ax_B.set_ylim(ymin, ymax)
# Titoli giustificati a sinistra
for ax, title in zip([ax_A, ax_B], ['A.\u2003FluSight', 'B.\u2003CovidHub']):
    ax.set_title(title, fontproperties=font_prop_title, pad=10, loc='left')

plt.savefig(f"../model-evaluation/figures/plot_locations/heatmap_locations_{metric_label}_log.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for col in df_plot_covid.columns:
    print(col)
    print(df_plot_covid[col].median())